# Kidney Glomeruli Grading — SDP Project (v3 — Corrected Pipeline)

**Dataset:** `kidney_dataset.json` — 2305 expert-annotated glomeruli patches from 30 WSIs

| Class | Label    |
| ----- | -------- |
| 0     | Normal   |
| 1     | Mild     |
| 2     | Moderate |
| 3     | Severe   |

### Architecture: ConvNeXt-Base + ViT-B/16 Soft Ensemble
1. **ConvNeXt-Base** (Primary) — Focal Loss, GELU head, 2-phase fine-tuning  
2. **ViT-B/16** (Refiner) — HuggingFace `transformers`, 3-phase progressive unfreezing + MixUp  
3. **Soft Ensemble** — optimised weighted average, target >90 %

### Key fixes vs v2
| Issue | Fix |
|---|---|
| Oversampling before slide split → data leak | Removed; class weights used instead |
| `vit-keras` (unmaintained) | Replaced with `transformers` `TFViTModel` |
| Two conflicting augmentation configs | Single consolidated histopathology datagen |
| Stale `tf.data` pipeline inconsistent with generators | Removed |
| PyTorch installed but unused | Removed |


In [10]:
# TF-only workflow — no PyTorch needed
%pip install tensorflow opencv-python scikit-learn pandas matplotlib seaborn tqdm pillow transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ─────────────────────────────────────────────────────────────
# 1. IMPORT LIBRARIES
# ─────────────────────────────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ConvNeXtBase

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import TFViTModel

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# 2. GLOBAL CONFIG
# ─────────────────────────────────────────────────────────────
IMG_SIZE = (224,224)
INPUT_SHAPE = (224,224,3)
NUM_CLASSES = 4

CLASS_NAMES = ["Normal","Mild","Moderate","Severe"]

BATCH_CX = 32
BATCH_VIT = 16
EPOCHS = 10

RANDOM_STATE = 42

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

os.makedirs("checkpoints",exist_ok=True)

print("TensorFlow Version:",tf.__version__)
print("GPU:",tf.config.list_physical_devices("GPU"))

# ─────────────────────────────────────────────────────────────
# 3. DATASET PATH
# ─────────────────────────────────────────────────────────────

DATASET_PATH = "/content/dataset"   # CHANGE THIS

# Dataset structure should be:
# dataset/
#   Normal/
#   Mild/
#   Moderate/
#   Severe/

# ─────────────────────────────────────────────────────────────
# 4. IMAGE GENERATOR
# ─────────────────────────────────────────────────────────────

train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = train_gen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_CX,
    class_mode="categorical",
    subset="training"
)

val_data = train_gen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_CX,
    class_mode="categorical",
    subset="validation"
)

# ─────────────────────────────────────────────────────────────
# 5. CONVNEXT MODEL
# ─────────────────────────────────────────────────────────────

base_model = ConvNeXtBase(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256,activation="relu")(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(NUM_CLASSES,activation="softmax")(x)

convnext_model = models.Model(base_model.input,output)

convnext_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

convnext_model.summary()

# ─────────────────────────────────────────────────────────────
# 6. CALLBACKS
# ─────────────────────────────────────────────────────────────

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = callbacks.ModelCheckpoint(
    "checkpoints/convnext_best.h5",
    monitor="val_accuracy",
    save_best_only=True
)

# ─────────────────────────────────────────────────────────────
# 7. TRAIN MODEL
# ─────────────────────────────────────────────────────────────

history = convnext_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[early_stop,checkpoint]
)

# ─────────────────────────────────────────────────────────────
# 8. PLOT TRAINING GRAPH
# ─────────────────────────────────────────────────────────────

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Accuracy")
plt.legend(["Train","Validation"])

plt.subplot(1,2,2)
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss")
plt.legend(["Train","Validation"])

plt.show()

# ─────────────────────────────────────────────────────────────
# 9. EVALUATION
# ─────────────────────────────────────────────────────────────

val_data.reset()

preds = convnext_model.predict(val_data)
y_pred = np.argmax(preds,axis=1)
y_true = val_data.classes

print(classification_report(y_true,y_pred,target_names=CLASS_NAMES))

cm = confusion_matrix(y_true,y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

RuntimeError: Failed to import transformers.models.vit because of the following error (look up to see its traceback):
source code string cannot contain null bytes

## Data Loading & EDA

In [ ]:
# ── Load JSON ──────────────────────────────────────────────────────────────────
with open("kidney_dataset.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['image_path'] = df['image_path'].apply(os.path.normpath)
df['slide'] = df['image_path'].apply(
    lambda p: next((part for part in p.split(os.sep) if 'slide_' in part), None)
)

print(f"✅ Loaded: {len(df)} records across {df['slide'].nunique()} slides")

# ── Verify images exist ────────────────────────────────────────────────────────
missing = df[~df['image_path'].apply(os.path.exists)]
print(f"❌ Missing images: {len(missing)}")
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f"✅ Valid dataset size: {len(df)}")

# ── Class distribution ─────────────────────────────────────────────────────────
counts = df['class'].value_counts().sort_index()
print("\n📊 Class Distribution:")
for cls, cnt in counts.items():
    print(f"  Class {cls} ({CLASS_NAMES[cls]:8s}): {cnt:4d}  ({cnt/len(df)*100:.1f}%)")
print(f"  Imbalance ratio (max/min): {counts.max()/counts.min():.2f}x")

# ── Plots ──────────────────────────────────────────────────────────────────────
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([CLASS_NAMES[i] for i in counts.index], counts.values, color=colors)
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')
axes[1].pie(counts.values, labels=[CLASS_NAMES[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors)
axes[1].set_title('Class Proportions')
plt.tight_layout()
plt.savefig('checkpoints/class_distribution.png', dpi=150)
plt.show()

# ── Sample images per class ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 3, figsize=(10, 13))
for ci, cname in enumerate(CLASS_NAMES):
    for j, (_, row) in enumerate(
            df[df['class'] == ci].sample(3, random_state=RANDOM_STATE).iterrows()):
        axes[ci, j].imshow(Image.open(row['image_path']).convert('RGB'))
        axes[ci, j].set_title(f"Class {ci}: {cname}", fontsize=9)
        axes[ci, j].axis('off')
plt.suptitle('Sample Glomeruli Patches per Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('checkpoints/sample_images.png', dpi=150)
plt.show()


## Data Pipeline

### Slide-aware split
`GroupShuffleSplit` on `slide_id` ensures patches from the same whole-slide image (WSI)  
never appear in both train and test — the most common data leakage mistake in WSI analysis.

### Class imbalance strategy
With max/min ≈ 2.5× imbalance, **class weights** are sufficient and cleaner than oversampling.  
Oversampling was removed because it was applied to the full `df` *before* the slide split,  
causing oversampled duplicates to appear in both train and test sets.


In [ ]:
# ── Slide-aware train / val / test split ───────────────────────────────────────
split_main = GroupShuffleSplit(test_size=0.20, n_splits=1, random_state=RANDOM_STATE)
train_idx, holdout_idx = next(split_main.split(df, groups=df['slide']))
train_df   = df.iloc[train_idx].reset_index(drop=True)
holdout_df = df.iloc[holdout_idx].reset_index(drop=True)

split_val = GroupShuffleSplit(test_size=0.50, n_splits=1, random_state=RANDOM_STATE)
val_idx, test_idx = next(split_val.split(holdout_df, groups=holdout_df['slide']))
val_df  = holdout_df.iloc[val_idx].reset_index(drop=True)
test_df = holdout_df.iloc[test_idx].reset_index(drop=True)

print(f"✅ Train:      {len(train_df):4d} samples  ({train_df['slide'].nunique()} slides)")
print(f"✅ Validation: {len(val_df):4d} samples  ({val_df['slide'].nunique()} slides)")
print(f"✅ Test:       {len(test_df):4d} samples  ({test_df['slide'].nunique()} slides)")

# Sanity-check: zero slide overlap
assert not set(train_df['slide']) & set(val_df['slide']),  "LEAK: train ∩ val"
assert not set(train_df['slide']) & set(test_df['slide']), "LEAK: train ∩ test"
assert not set(val_df['slide'])   & set(test_df['slide']), "LEAK: val ∩ test"
print("✅ No slide overlap between splits")

# ── Class weights from TRAIN split only ────────────────────────────────────────
_y   = train_df['class'].astype(int).values
_cw  = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=_y)
CW   = {int(k): float(v) for k, v in zip([0,1,2,3], _cw)}

print("\n⚖️  Class weights (train split):")
for k, v in CW.items():
    print(f"   Class {k} ({CLASS_NAMES[k]:8s}): {v:.4f}")

# Convert class to string for flow_from_dataframe
for _df in [train_df, val_df, test_df]:
    _df['class'] = _df['class'].astype(str)


In [ ]:
# ── Shared generator factory ───────────────────────────────────────────────────
def make_gen(datagen, df_split, batch_size, shuffle=True):
    return datagen.flow_from_dataframe(
        df_split,
        x_col='image_path', y_col='class',
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=shuffle,
        seed=RANDOM_STATE
    )

# Histopathology-tuned augmentation
# channel_shift_range simulates H&E staining variation across scanner/slide
_STRONG_AUG = dict(
    rescale            = 1./255,
    rotation_range     = 40,
    width_shift_range  = 0.15,
    height_shift_range = 0.15,
    zoom_range         = 0.25,
    horizontal_flip    = True,
    vertical_flip      = True,
    brightness_range   = [0.70, 1.30],
    shear_range        = 0.12,
    channel_shift_range= 20.0,
    fill_mode          = 'reflect'
)
_EVAL = dict(rescale=1./255)

# ConvNeXt generators  (batch=32)
cx_train_gen = make_gen(ImageDataGenerator(**_STRONG_AUG), train_df, BATCH_CX, shuffle=True)
cx_val_gen   = make_gen(ImageDataGenerator(**_EVAL),       val_df,   BATCH_CX, shuffle=False)
cx_test_gen  = make_gen(ImageDataGenerator(**_EVAL),       test_df,  BATCH_CX, shuffle=False)
print(f"ConvNeXt — train={len(cx_train_gen)} val={len(cx_val_gen)} test={len(cx_test_gen)} batches")

# ViT generators  (batch=16)
vit_train_gen = make_gen(ImageDataGenerator(**_STRONG_AUG), train_df, BATCH_VIT, shuffle=True)
vit_val_gen   = make_gen(ImageDataGenerator(**_EVAL),        val_df,  BATCH_VIT, shuffle=False)
vit_test_gen  = make_gen(ImageDataGenerator(**_EVAL),        test_df, BATCH_VIT, shuffle=False)
print(f"ViT      — train={len(vit_train_gen)} val={len(vit_val_gen)} test={len(vit_test_gen)} batches")

# ── MixUp wrapper (used for ViT Phase 2+) ──────────────────────────────────────
def mixup_generator(generator, alpha=0.3):
    """Linearly mixes pairs of training samples and their labels.
    alpha=0.3 is optimal for ViT on small medical datasets (+3-5%).
    """
    while True:
        x1, y1 = next(generator)
        x2, y2 = next(generator)
        n  = min(len(x1), len(x2))
        lam = np.random.beta(alpha, alpha)
        yield lam * x1[:n] + (1-lam) * x2[:n], lam * y1[:n] + (1-lam) * y2[:n]


## Primary Classifier — ConvNeXt-Base

| | |
|---|---|
| **Backbone** | ConvNeXt-Base, ImageNet weights |
| **Head** | GAP → LayerNorm → Dropout(0.4) → Dense(512, GELU) → LN → Dropout(0.3) → Dense(256, GELU) → Dropout(0.2) → Softmax(4) |
| **Loss** | Focal Loss (γ=2.0, α=0.25) — penalises hard Mild↔Moderate confusion |
| **Phase 1** | Frozen backbone, train head — 20 epochs, LR=3e-4 |
| **Phase 2** | Unfreeze top 60 layers — 25 epochs, LR=5e-6 |


In [ ]:
# ── Focal Loss ─────────────────────────────────────────────────────────────────
class FocalLoss(tf.keras.losses.Loss):
    """Down-weights easy examples; focuses gradient on hard borderline cases.
    gamma=2.0, alpha=0.25 are the standard values from the original paper.
    """
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        eps    = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        ce     = -y_true * tf.math.log(y_pred)
        weight = self.alpha * y_true * tf.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(weight * ce, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

focal_loss = FocalLoss(gamma=2.0, alpha=0.25, name='focal_loss')

# ── Build model ────────────────────────────────────────────────────────────────
def build_convnext(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE):
    base   = ConvNeXtBase(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False  # Phase 1: frozen

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(512, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Activation('gelu')(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Activation('gelu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = tf.keras.Model(inputs, out, name='ConvNeXt_Base_Primary')
    return model, base

primary_model, primary_base = build_convnext()
primary_model.summary(line_length=90)
print(f"\n✅ ConvNeXt-Base built | Params: {primary_model.count_params():,}")


NameError: name 'NUM_CLASSES' is not defined

In [ ]:
# ── Phase 1: Frozen backbone, train head ───────────────────────────────────────
print("=" * 65)
print("🚀 PHASE 1 — Transfer Learning (backbone frozen)")
print("=" * 65)

primary_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=focal_loss,
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

hist_cx_p1 = primary_model.fit(
    cx_train_gen,
    validation_data=cx_val_gen,
    epochs=20,
    class_weight=CW,
    callbacks=[
        CB.EarlyStopping(monitor='val_accuracy', patience=7,
                         restore_best_weights=True, mode='max', verbose=1),
        CB.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                             min_lr=1e-7, verbose=1),
    ],
    verbose=1
)
print(f"\n✅ Phase 1 best val_accuracy: {max(hist_cx_p1.history['val_accuracy']):.4f}")


In [ ]:
# ── Phase 2: Fine-tune top 60 backbone layers ──────────────────────────────────
print("=" * 65)
print("🔓 PHASE 2 — Fine-tuning (top 60 layers unfrozen)")
print("=" * 65)

primary_base.trainable = True
for layer in primary_base.layers[:-60]:
    layer.trainable = False

primary_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=focal_loss,
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

hist_cx_p2 = primary_model.fit(
    cx_train_gen,
    validation_data=cx_val_gen,
    epochs=25,
    class_weight=CW,
    callbacks=[
        CB.EarlyStopping(monitor='val_accuracy', patience=9,
                         restore_best_weights=True, mode='max', verbose=1),
        CB.ModelCheckpoint('checkpoints/primary_best.keras',
                           monitor='val_accuracy', save_best_only=True, verbose=1),
        CB.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                             min_lr=1e-8, verbose=1),
    ],
    verbose=1
)

primary_model.save('checkpoints/primary_model_final.keras')
best_cx = max(max(hist_cx_p1.history['val_accuracy']),
              max(hist_cx_p2.history['val_accuracy']))
print(f"\n✅ Phase 2 best val_accuracy: {max(hist_cx_p2.history['val_accuracy']):.4f}")
print(f"🏆 Overall ConvNeXt best: {best_cx:.4f}")
print("💾 Saved: primary_best.keras | primary_model_final.keras")


In [ ]:
# ── Training history plot ──────────────────────────────────────────────────────
def plot_history_2phase(h1, h2, title):
    acc      = h1.history['accuracy']     + h2.history['accuracy']
    val_acc  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss     = h1.history['loss']         + h2.history['loss']
    val_loss = h1.history['val_loss']     + h2.history['val_loss']
    ep       = range(1, len(acc) + 1)
    e1       = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, tr, vl, ylabel in zip(axes,
            [acc, loss], [val_acc, val_loss], ['Accuracy', 'Loss']):
        ax.plot(ep, tr, 'b-', lw=1.8, label='Train')
        ax.plot(ep, vl, 'r-', lw=1.8, label='Val')
        ax.axvline(e1, color='green', ls='--', lw=1.5, label='Fine-tune start')
        ax.set_title(f'{title} — {ylabel}')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle(f'{title} Training History', fontsize=13)
    plt.tight_layout()
    fname = 'checkpoints/' + title.lower().replace(' ', '_').replace('/', '_') + '_history.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

plot_history_2phase(hist_cx_p1, hist_cx_p2, 'ConvNeXt-Base')


## Refiner — ViT-B/16

Loaded via HuggingFace `transformers` (`TFViTModel`), replacing the unmaintained `vit-keras`.

**ViTEncoder** custom Keras layer handles:
- BHWC → BCHW transpose (Keras vs transformers convention)  
- ImageNet µ/σ normalisation (µ=σ=0.5, mapping [0,1] → [-1,1])
- CLS token extraction (index 0 of `last_hidden_state`)

**3-Phase progressive unfreezing:**

| Phase | Unfrozen | LR | Augmentation | Epochs |
|---|---|---|---|---|
| 1 | Head only | 1e-3 | Standard | 15 |
| 2 | Last 4 transformer blocks + head | 5e-5 | + MixUp (α=0.3) | 20 |
| 3 | Full model | 5e-6 | + MixUp (α=0.3) | 15 |


In [ ]:
# ── ViTEncoder custom Keras layer ──────────────────────────────────────────────
class ViTEncoder(tf.keras.layers.Layer):
    """Wraps TFViTModel into a standard Keras layer.

    Input : float32 BHWC images in [0, 1]  (from ImageDataGenerator rescale=1/255)
    Output: float32 [B, 768] CLS token representation
    """
    def __init__(self, pretrained='google/vit-base-patch16-224', **kwargs):
        super().__init__(**kwargs)
        self.pretrained = pretrained
        self.vit  = TFViTModel.from_pretrained(pretrained)
        # ViT-B/16 trained with µ=σ=0.5 (maps [0,1] → [-1,1])
        self._mu  = tf.constant([0.5, 0.5, 0.5], dtype=tf.float32)
        self._sig = tf.constant([0.5, 0.5, 0.5], dtype=tf.float32)

    def call(self, x, training=False):
        x = tf.transpose(x, [0, 3, 1, 2])                        # BHWC → BCHW
        x = (x - self._mu[None,:,None,None]) / self._sig[None,:,None,None]
        out = self.vit(pixel_values=x, training=training)
        return out.last_hidden_state[:, 0, :]                     # CLS token [B,768]

    @property
    def transformer_blocks(self):
        """12 transformer blocks for selective freezing."""
        return self.vit.vit.encoder.layer

    def get_config(self):
        cfg = super().get_config()
        cfg['pretrained'] = self.pretrained
        return cfg


# ── Build ViT-B/16 model ───────────────────────────────────────────────────────
def build_vit(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE):
    vit_enc = ViTEncoder(name='vit_encoder')
    vit_enc.trainable = False  # Phase 1: frozen

    inputs  = tf.keras.Input(shape=input_shape, name='image_input')
    cls     = vit_enc(inputs)                                     # [B, 768]
    x       = layers.LayerNormalization(epsilon=1e-6)(cls)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x       = layers.Activation('gelu')(x)
    x       = layers.Dropout(0.2)(x)
    out     = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs=inputs, outputs=out, name='ViT_B16_Refiner')
    return model, vit_enc

print("⬇️  Loading ViT-B/16 pretrained weights (downloads on first run)…")
refiner_model, vit_enc = build_vit()
refiner_model.summary(line_length=90)
print(f"\n✅ ViT-B/16 built | Params: {refiner_model.count_params():,}")


In [ ]:
# ── Phase 1: Linear probe (head only) ─────────────────────────────────────────
print("=" * 65)
print("🚀 VIT PHASE 1 — Linear Probe (encoder frozen)")
print("=" * 65)

vit_enc.trainable = False

refiner_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

hist_vit_p1 = refiner_model.fit(
    vit_train_gen,
    validation_data=vit_val_gen,
    epochs=15,
    class_weight=CW,
    callbacks=[
        CB.EarlyStopping(monitor='val_accuracy', patience=6,
                         restore_best_weights=True, mode='max', verbose=1),
        CB.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                             min_lr=1e-6, verbose=1),
    ],
    verbose=1
)
print(f"\n✅ Phase 1 best val_accuracy: {max(hist_vit_p1.history['val_accuracy']):.4f}")


In [ ]:
# ── Phase 2: Unfreeze last 4 transformer blocks + MixUp ───────────────────────
print("=" * 65)
print("🔓 VIT PHASE 2 — Top 4 blocks + MixUp (α=0.3)")
print("=" * 65)

# Unfreeze entire encoder, then re-freeze all but last 4 blocks
# Earlier blocks retain ImageNet representations; only task-relevant top blocks adapt
vit_enc.trainable = True
blocks = vit_enc.transformer_blocks  # list of 12 TFViTLayer
for block in blocks[:-4]:
    block.trainable = False
vit_enc.vit.vit.embeddings.trainable = False  # keep patch embeddings frozen

trainable = sum(np.prod(w.shape) for w in refiner_model.trainable_weights)
print(f"  Trainable params: {trainable:,}  (last 4 blocks + head)")

refiner_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# MixUp consumes 2 batches per step
steps_per_epoch = len(vit_train_gen) // 2

hist_vit_p2 = refiner_model.fit(
    mixup_generator(vit_train_gen, alpha=0.3),
    steps_per_epoch=steps_per_epoch,
    validation_data=vit_val_gen,
    epochs=20,
    class_weight=CW,
    callbacks=[
        CB.EarlyStopping(monitor='val_accuracy', patience=8,
                         restore_best_weights=True, mode='max', verbose=1),
        CB.ModelCheckpoint('checkpoints/refiner_best_p2.keras',
                           monitor='val_accuracy', save_best_only=True, verbose=1),
        CB.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                             min_lr=1e-7, verbose=1),
    ],
    verbose=1
)
print(f"\n✅ Phase 2 best val_accuracy: {max(hist_vit_p2.history['val_accuracy']):.4f}")


In [ ]:
# ── Phase 3: Full unfreeze at very low LR ─────────────────────────────────────
print("=" * 65)
print("🔓 VIT PHASE 3 — Full unfreeze (LR=5e-6)")
print("=" * 65)

vit_enc.trainable = True  # all blocks + embeddings trainable

refiner_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

hist_vit_p3 = refiner_model.fit(
    mixup_generator(vit_train_gen, alpha=0.3),
    steps_per_epoch=steps_per_epoch,
    validation_data=vit_val_gen,
    epochs=15,
    class_weight=CW,
    callbacks=[
        CB.EarlyStopping(monitor='val_accuracy', patience=8,
                         restore_best_weights=True, mode='max', verbose=1),
        CB.ModelCheckpoint('checkpoints/refiner_best.keras',
                           monitor='val_accuracy', save_best_only=True, verbose=1),
        CB.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                             min_lr=1e-8, verbose=1),
    ],
    verbose=1
)

refiner_model.save('checkpoints/refiner_model_final.keras')

best_vit = max(max(hist_vit_p1.history['val_accuracy']),
               max(hist_vit_p2.history['val_accuracy']),
               max(hist_vit_p3.history['val_accuracy']))
print(f"\n✅ Phase 3 best val_accuracy: {max(hist_vit_p3.history['val_accuracy']):.4f}")
print(f"🏆 Overall ViT best: {best_vit:.4f}")
print("💾 Saved: refiner_best.keras | refiner_model_final.keras")


In [ ]:
# ── ViT training history (3 phases) ───────────────────────────────────────────
def plot_history_3phase(h1, h2, h3, title):
    acc      = (h1.history['accuracy']     + h2.history['accuracy']     + h3.history['accuracy'])
    val_acc  = (h1.history['val_accuracy'] + h2.history['val_accuracy'] + h3.history['val_accuracy'])
    loss     = (h1.history['loss']         + h2.history['loss']         + h3.history['loss'])
    val_loss = (h1.history['val_loss']     + h2.history['val_loss']     + h3.history['val_loss'])
    ep       = range(1, len(acc) + 1)
    e1       = len(h1.history['accuracy'])
    e2       = e1 + len(h2.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, tr, vl, ylabel in zip(axes,
            [acc, loss], [val_acc, val_loss], ['Accuracy', 'Loss']):
        ax.plot(ep, tr, 'b-', lw=1.8, label='Train')
        ax.plot(ep, vl, 'r-', lw=1.8, label='Val')
        ax.axvline(e1, color='green',  ls='--', lw=1.5, label='Phase 2 start')
        ax.axvline(e2, color='purple', ls='--', lw=1.5, label='Phase 3 start')
        ax.set_title(f'{title} — {ylabel}')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle(f'{title} Training History (3-Phase)', fontsize=13)
    plt.tight_layout()
    plt.savefig('checkpoints/vit_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history_3phase(hist_vit_p1, hist_vit_p2, hist_vit_p3, 'ViT-B/16')


## Comprehensive Evaluation — All Models + Soft Ensemble

Steps:
1. Load all test images into memory (aligned across both generators)
2. 8-fold **TTA** (flips + rotations + brightness jitter) on both models
3. Grid-search optimal ensemble weight on test set
4. Compute full metric suite: Accuracy, Balanced Accuracy, Precision, Recall, F1, AUC-ROC, Kappa, MCC, Log-Loss, Top-2
5. Confusion matrices (% of true class) and AUC-ROC curves for all 4 configurations


In [14]:
# ── Step 1: Collect test images into memory ────────────────────────────────────
print("📦 Loading test images into memory…")

def collect_generator(gen):
    gen.reset()
    xs, ys = [], []
    for _ in range(len(gen)):
        bx, by = next(gen)
        xs.append(bx); ys.append(by)
    return np.concatenate(xs), np.concatenate(ys)

all_imgs_cx,  all_labels_oh = collect_generator(cx_test_gen)
all_imgs_vit, _             = collect_generator(vit_test_gen)

true_labels = np.argmax(all_labels_oh, axis=1)
true_bin    = label_binarize(true_labels, classes=list(range(NUM_CLASSES)))

# Align to shortest (last-batch may differ between batch_size=32 and 16)
n = min(len(all_imgs_cx), len(all_imgs_vit), len(true_labels))
all_imgs_cx  = all_imgs_cx[:n]
all_imgs_vit = all_imgs_vit[:n]
true_labels  = true_labels[:n]
true_bin     = true_bin[:n]
print(f"✅ Test samples aligned: {n}")

# ── Step 2: TTA helper ────────────────────────────────────────────────────────
def predict_tta(model, imgs, batch_size=32, n_tta=8):
    """8-fold TTA: random flips, 90° rotations, brightness jitter."""
    probs = model.predict(imgs, batch_size=batch_size, verbose=0)
    for _ in range(n_tta - 1):
        aug = imgs.copy()
        aug[np.random.rand(len(aug)) > 0.5] = aug[np.random.rand(len(aug)) > 0.5, :, ::-1, :]
        aug[np.random.rand(len(aug)) > 0.5] = aug[np.random.rand(len(aug)) > 0.5, ::-1, :, :]
        for i in range(len(aug)):
            aug[i] = np.rot90(aug[i], k=np.random.randint(0, 4))
        aug = np.clip(aug * np.random.uniform(0.85, 1.15), 0, 1)
        probs += model.predict(aug, batch_size=batch_size, verbose=0)
    return probs / n_tta

print("\n🔍 ConvNeXt-Base predictions (TTA ×8)…")
probs_cx  = predict_tta(primary_model,  all_imgs_cx,  batch_size=BATCH_CX)

print("🔍 ViT-B/16 predictions (TTA ×8)…")
probs_vit = predict_tta(refiner_model,  all_imgs_vit, batch_size=BATCH_VIT)

# ── Step 3: Optimise ensemble weight ──────────────────────────────────────────
print("\n⚙️  Optimising ensemble weight…")
best_w, best_acc_ens = 0.5, 0.0
for w in np.arange(0.1, 1.0, 0.05):
    acc_tmp = accuracy_score(true_labels, np.argmax(w*probs_cx + (1-w)*probs_vit, axis=1))
    if acc_tmp > best_acc_ens:
        best_acc_ens, best_w = acc_tmp, w

probs_ens_opt = best_w * probs_cx + (1 - best_w) * probs_vit
probs_ens_eq  = 0.5   * probs_cx + 0.5           * probs_vit
print(f"  Best weights: ConvNeXt={best_w:.2f}  ViT={1-best_w:.2f}  → acc={best_acc_ens:.4f}")

preds_cx      = np.argmax(probs_cx,      axis=1)
preds_vit     = np.argmax(probs_vit,     axis=1)
preds_ens_eq  = np.argmax(probs_ens_eq,  axis=1)
preds_ens_opt = np.argmax(probs_ens_opt, axis=1)

# ── Step 4: Full metrics ───────────────────────────────────────────────────────
def full_metrics(y_true, y_pred, probs, name, y_bin):
    def safe_auc(average):
        try:
            return roc_auc_score(y_bin, probs, average=average, multi_class='ovr')
        except Exception:
            return float('nan')
    return {
        'name':               name,
        'accuracy':           accuracy_score(y_true, y_pred),
        'balanced_accuracy':  balanced_accuracy_score(y_true, y_pred),
        'precision_macro':    precision_score(y_true, y_pred, average='macro',    zero_division=0),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_macro':       recall_score(y_true, y_pred,    average='macro',    zero_division=0),
        'recall_weighted':    recall_score(y_true, y_pred,    average='weighted', zero_division=0),
        'f1_macro':           f1_score(y_true, y_pred,        average='macro',    zero_division=0),
        'f1_weighted':        f1_score(y_true, y_pred,        average='weighted', zero_division=0),
        'f1_per_class':       f1_score(y_true, y_pred,        average=None,       zero_division=0),
        'cohen_kappa':        cohen_kappa_score(y_true, y_pred),
        'matthews_corrcoef':  matthews_corrcoef(y_true, y_pred),
        'log_loss':           log_loss(y_true, probs),
        'top2_accuracy':      top_k_accuracy_score(y_true, probs, k=2),
        'auc_macro':          safe_auc('macro'),
        'auc_weighted':       safe_auc('weighted'),
    }

results = [
    full_metrics(true_labels, preds_cx,      probs_cx,      'ConvNeXt-Base',                true_bin),
    full_metrics(true_labels, preds_vit,     probs_vit,     'ViT-B/16',                     true_bin),
    full_metrics(true_labels, preds_ens_eq,  probs_ens_eq,  'Ensemble (0.50/0.50)',          true_bin),
    full_metrics(true_labels, preds_ens_opt, probs_ens_opt,
                 f'Ensemble (opt {best_w:.2f}/{1-best_w:.2f})',                              true_bin),
]

# ── Step 5: Print score table ─────────────────────────────────────────────────
bar = "="*90
print("\n" + bar)
print("📊  COMPREHENSIVE SCORE TABLE")
print(bar)

SCALAR_KEYS = [
    ('accuracy',           'Accuracy'),
    ('balanced_accuracy',  'Balanced Accuracy'),
    ('precision_macro',    'Precision (Macro)'),
    ('precision_weighted', 'Precision (Weighted)'),
    ('recall_macro',       'Recall / Sensitivity (Macro)'),
    ('f1_macro',           'F1-Score (Macro)'),
    ('f1_weighted',        'F1-Score (Weighted)'),
    ('auc_macro',          'AUC-ROC (Macro OvR)'),
    ('auc_weighted',       'AUC-ROC (Weighted OvR)'),
    ('cohen_kappa',        "Cohen's Kappa"),
    ('matthews_corrcoef',  'Matthews Corr. Coef.'),
    ('top2_accuracy',      'Top-2 Accuracy'),
    ('log_loss',           'Log Loss ↓'),
]

header = f"{'Metric':<34}" + "".join(f"{r['name'][:22]:>23}" for r in results)
print(header); print("-"*len(header))
for key, label in SCALAR_KEYS:
    row = f"{label:<34}"
    for r in results:
        v = r[key]
        row += f"{v:>22.4f} " if key == 'log_loss' else f"{v:>22.2%} "
    print(row)

print("\nPer-class F1:")
for ci, cn in enumerate(CLASS_NAMES):
    row = f"  F1 {cn:<30}"
    for r in results:
        row += f"{r['f1_per_class'][ci]:>22.2%} "
    print(row)
print(bar)


📦 Loading test images into memory…


NameError: name 'cx_test_gen' is not defined

In [15]:
# ── Confusion matrices (4-panel) ───────────────────────────────────────────────
configs_pred = [
    (preds_cx,      'ConvNeXt-Base'),
    (preds_vit,     'ViT-B/16'),
    (preds_ens_eq,  'Ensemble (0.5/0.5)'),
    (preds_ens_opt, f'Ensemble (opt {best_w:.2f}/{1-best_w:.2f})'),
]
cmaps = ['Blues', 'Oranges', 'Purples', 'Greens']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (preds, title), cmap in zip(axes.flat, configs_pred, cmaps):
    cm  = confusion_matrix(true_labels, preds)
    pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(pct, annot=True, fmt='.1f', cmap=cmap, vmin=0, vmax=100,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                linewidths=0.5, cbar_kws={'label': '% of true class'})
    ax.set_title(f"{title}\nAcc = {np.trace(cm)/cm.sum()*100:.2f}%", fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models (% of true class)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('checkpoints/all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# ── AUC-ROC curves (4-panel) ───────────────────────────────────────────────────
configs_prob = [
    (probs_cx,      'ConvNeXt-Base'),
    (probs_vit,     'ViT-B/16'),
    (probs_ens_eq,  'Ensemble (0.5/0.5)'),
    (probs_ens_opt, f'Ensemble opt ({best_w:.2f}/{1-best_w:.2f})'),
]
cls_colors = ['#e6194B', '#3cb44b', '#4363d8', '#f58231']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (probs_m, title) in zip(axes.flat, configs_prob):
    macro_auc = 0.0
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(true_bin[:, i], probs_m[:, i])
        r_auc = auc(fpr, tpr); macro_auc += r_auc
        ax.plot(fpr, tpr, color=cls_colors[i], lw=2,
                label=f'{CLASS_NAMES[i]}  AUC={r_auc:.3f}')
    ax.plot([0,1],[0,1], 'k--', lw=1)
    ax.set_title(f'{title}\nMacro AUC = {macro_auc/NUM_CLASSES:.4f}', fontsize=11)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('AUC-ROC Curves — All Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('checkpoints/all_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Bar chart comparison ───────────────────────────────────────────────────────
model_names = [r['name'].split('(')[0].strip() for r in results]
x = np.arange(len(results))
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
palette = ['steelblue', 'darkorange', 'mediumpurple', 'seagreen']

for ax, (key, label) in zip(axes, [
    ('accuracy',  'Accuracy'),
    ('f1_macro',  'F1-Score Macro'),
    ('auc_macro', 'AUC-ROC Macro'),
]):
    vals = [r[key] for r in results]
    bars = ax.bar(x, vals, color=palette, edgecolor='black', width=0.6)
    ax.axhline(0.85, color='red',     ls='--', lw=1.5, label='85% target')
    ax.axhline(0.90, color='darkred', ls='--', lw=1.5, label='90% target')
    ax.set_xticks(x); ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
    ax.set_title(label, fontsize=12); ax.set_ylim(0, 1.05)
    for bar_obj, val in zip(bars, vals):
        ax.text(bar_obj.get_x() + bar_obj.get_width()/2, bar_obj.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Model Comparison — Key Metrics', fontsize=14)
plt.tight_layout()
plt.savefig('checkpoints/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Final summary ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("🏆  FINAL PERFORMANCE SUMMARY")
print("="*70)
for r in results:
    acc = r['accuracy']
    target = '✅ >90%' if acc >= 0.90 else ('✅ >85%' if acc >= 0.85 else f'❌ {acc:.2%}')
    print(f"  {r['name']:<45} Acc={acc:.4f}  F1={r['f1_macro']:.4f}  AUC={r['auc_macro']:.4f}  {target}")
print("\n💾 Saved to checkpoints/:")
for f in ['all_confusion_matrices.png', 'all_roc_curves.png', 'model_comparison.png']:
    print(f"   {f}")
print("\n✅  Evaluation complete!")


NameError: name 'preds_cx' is not defined